In [0]:
table_name = "dim_cpt_codes"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_cpt_codes")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import upper, trim, col, abs as spark_abs

# 1. Standardize text: UPPER(TRIM()) for text columns
df = df.withColumn("cpt_code", upper(trim(col("cpt_code"))))
df = df.withColumn("cpt_description", upper(trim(col("cpt_description"))))
df = df.withColumn("cpt_category", upper(trim(col("cpt_category"))))

# 2. Precision Optimization: Safe Cast to DECIMAL(15,2). If < 0, apply ABS()
for fin_col in ["base_price", "reimbursement_rate", "expected_charge"]:
    df = df.withColumn(fin_col, trim(col(fin_col)).cast("decimal(15,2)"))
    df = df.withColumn(fin_col, spark_abs(col(fin_col)))

# 3. Drop audit columns and source surrogate key
df = df.drop("_ingested_at", "_source_file", "cpt_code_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")